[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)

# Evaluate, Experiment, Improve

Notebook 2 made the agent's behavior visible. Now we will evaluate a few important skills,
inspect failures in Phoenix, and test a prompt change against the same examples.

**Learning path:** run a small baseline → inspect router and skill scores → change one prompt
→ compare the same cases → review a failure → decide whether it belongs in the golden dataset.

The short lab uses three cases to keep live calls manageable. The complete 11-case suite stays
in the shared harness and CI.


---
## 1. Install the workshop and Phoenix client packages

This mirrors Notebook 2's Phoenix setup. The board is hosted; this installs only the client
and tracing SDKs, not a second Phoenix server. The base packages let this notebook run on its
own Colab runtime too.


In [ ]:
%pip install -q "openai>=1.40" "pandas>=2.0" "matplotlib>=3.8" arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install before importing the agent or tracing SDK. If Colab says the runtime
> needs a restart, restart it and rerun the notebook from the top.


---
## 2. Load the same agent and credentials as Notebook 2

Notebook 3 imports the canonical agent from `utils.py`, loads the frozen golden cases, and
uses the same gateway and seat key as Notebook 2. The Phoenix board key is separate from the
workshop model key.


In [ ]:
import urllib.request
import os
import importlib
from pathlib import Path

utils_path = Path("utils.py")
required_utils = (
    "def set_span_tracer(tracer, attributes=None):",
    "def load_golden():",
    "def run_suite(label, system_prompt, temperature, agent_fn=None, judge_fn=None,",
)
if not utils_path.exists() or any(marker not in utils_path.read_text() for marker in required_utils):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils
utils = importlib.reload(utils)

utils.load_data()
utils.load_golden()
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_EVAL_PROJECT = "01-Part-3-evaluations"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from phoenix.client import Client
phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
try:
    phoenix_client.projects.list()
except Exception as exc:
    if "401" not in str(exc) and "Unauthorized" not in str(exc):
        raise
    from getpass import getpass
    print("Saved Phoenix key was rejected. Enter the participant board key supplied by the organizer.")
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
    phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
    phoenix_client.projects.list()

from utils import (
    AGENT_MODEL, CREATIVE_SYSTEM_PROMPT, HARDENED_SYSTEM_PROMPT, GOLDEN,
    REQUIRED_TOOLS, SUITE_RESULTS, check_code, check_numeric, check_router, run_agent, run_suite,
)

print("bootstrap done |", len(GOLDEN["cases"]), "golden cases | seat:", SEAT)


## 3. Send this notebook's traces to Phoenix

As in Notebook 2, the OTLP exporter writes directly to the shared board. Notebook 3 uses
`01-Part-3-evaluations`, separate from Notebook 2's trace-review project. The stable
`session.id` identifies your seat; unique run labels distinguish experiment variants.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=PHOENIX_EVAL_PROJECT,
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("Phoenix project:", PHOENIX_EVAL_PROJECT)
print("Phoenix board:", PHOENIX_BOARD_URL)


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes

openai_instrumentor = OpenAIInstrumentor()
if openai_instrumentor.is_instrumented_by_opentelemetry:
    openai_instrumentor.uninstrument()
openai_instrumentor.instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(
    tracer_provider.get_tracer(__name__),
    attributes={SpanAttributes.SESSION_ID: SEAT},
)
print("OpenAI and agent spans are instrumented")


---
## 4. Run a small baseline

An evaluation is a repeatable question, expected behavior, and evaluator. We will start with
three examples from the frozen golden dataset: signed expenses, merchant aliases, and a
missing schema field. The same checks and examples will be used for every variant. Each
variant gets a parent Phoenix span with an overall pass rate and separate router, tool, and
answer scores, similar to the per-evaluator summaries in L11.


In [ ]:
from IPython.display import display
import pandas as pd

STARTER_IDS = ["total_expenses_october", "starbucks_total", "merchant_location"]
STARTER_CASES = [case for case in GOLDEN["cases"] if case["id"] in STARTER_IDS]
if len(STARTER_CASES) != len(STARTER_IDS):
    raise ValueError("Starter cases are missing from the golden dataset")

import datetime as _datetime
RUN_TAG = _datetime.datetime.now(_datetime.timezone.utc).strftime("lab3-%Y%m%d-%H%M%S-%f")
BASELINE_LABEL = f"{RUN_TAG}/prompt-baseline"

EXPERIMENT_RUNS = []


def run_phoenix_experiment(label, system_prompt, cases, temperature=0.0, model=None):
    """Run one fixed case set under a named Phoenix parent span and score it."""
    span_attributes = {
        SpanAttributes.SESSION_ID: SEAT,
        "experiment.variant": label,
        "experiment.case_ids": [case["id"] for case in cases],
    }
    experiment_tracer = tracer_provider.get_tracer("notebook3.experiments")
    with experiment_tracer.start_as_current_span(
        f"experiment/{label}", attributes=span_attributes
    ) as experiment_span:
        scorecard = run_suite(
            label, system_prompt, temperature, cases=cases, model=model
        )

        router_score = float(scorecard["router_ok"].mean())
        code_rows = scorecard.loc[scorecard["n_chunks"].fillna(0) > 0]
        tool_score = (
            float((code_rows["code_valid"] & code_rows["no_traceback"]).mean())
            if len(code_rows) else None
        )
        answer_scores = []
        for row in scorecard.to_dict(orient="records"):
            numeric = row.get("numeric_ok")
            judged = row.get("judge_pass")
            if pd.notna(numeric):
                answer_scores.append(float(numeric))
            elif pd.notna(judged):
                answer_scores.append(float(judged))
        answer_score = sum(answer_scores) / len(answer_scores) if answer_scores else None
        overall_score = float(scorecard["PASS"].mean())
        passed = int(scorecard["PASS"].sum())

        component_scores = {
            "router": router_score,
            "tool": tool_score,
            "answer": answer_score,
            "overall": overall_score,
        }
        for name, value in component_scores.items():
            if value is not None:
                experiment_span.set_attribute(f"eval.{name}_score", value)
        experiment_span.set_attribute("eval.passed_cases", passed)
        experiment_span.set_attribute("eval.total_cases", len(scorecard))
        experiment_span_id = f"{experiment_span.get_span_context().span_id:016x}"

    run_record = {
        "label": label,
        "scorecard": scorecard,
        "span_id": experiment_span_id,
        "scores": component_scores,
        "passed": passed,
        "total": len(scorecard),
    }
    EXPERIMENT_RUNS.append(run_record)
    tool_text = f"{tool_score:.0%}" if tool_score is not None else "n/a"
    answer_text = f"{answer_score:.0%}" if answer_score is not None else "n/a"
    print(
        f"{label} scores | overall={overall_score:.0%} ({passed}/{len(scorecard)})"
        f" | router={router_score:.0%} | tool={tool_text} | answer={answer_text}"
    )
    return scorecard


baseline = run_phoenix_experiment(
    BASELINE_LABEL, HARDENED_SYSTEM_PROMPT, STARTER_CASES, temperature=0.0
)
display(baseline[[c for c in ("case", "router_ok", "tools_used", "code_valid", "no_traceback", "numeric_ok", "judge_pass", "PASS") if c in baseline]])


### What are the evaluators checking?

The full suite runs all applicable checks for every case. We'll call three of them separately
on one known case so the result of each evaluator is visible:

1. **Router evaluator** — did the agent call the required tool?
2. **Tool/code evaluator** — did the analysis tool generate valid code that ran successfully?
3. **Answer evaluator** — did the final answer match the frozen ground truth?

The missing-location case uses a separate judge for honesty and usefulness. A router pass does
not imply a tool or answer pass; Phoenix will show each score independently.

The analysis tool also cautions the code generator to inspect the full dataset for likely
entity variants and decoys, using bounded summaries instead of dumping rows. The merchant case
tests whether that general caution is enough; it is not a guarantee, so inspect the generated
code and trace evidence.


#### Evaluator 1 — router choice

This check reads the actual tool calls from the agent run and compares them with the tool
required by the golden case. It does not judge whether the tool's result is correct.


In [ ]:
example_case = next(c for c in STARTER_CASES if c["id"] == "total_expenses_october")
example_result = SUITE_RESULTS[BASELINE_LABEL][example_case["id"]]
router_eval = check_router(example_case, example_result)
router_eval["label"] = "correct" if router_eval["router_ok"] else "incorrect"
router_eval["score"] = 1 if router_eval["router_ok"] else 0
router_score = float(router_eval["score"])
print(router_eval)
print(f"Router score: {router_score:.0f}/1")


#### Evaluator 2 — analysis tool code

For code-producing cases, this checks syntax and whether execution raised an error. It is a
tool/skill check, not an answer-correctness check. Non-code cases do not receive this score.


In [ ]:
tool_eval = check_code(example_result)
tool_ok = bool(tool_eval["code_valid"] and tool_eval["no_traceback"])
tool_eval["label"] = "runnable" if tool_ok else "not_runnable"
tool_eval["score"] = 1 if tool_ok else 0
tool_score = float(tool_eval["score"])
print(tool_eval)
print(f"Tool/code score: {tool_score:.0f}/1")


#### Evaluator 3 — answer against ground truth

This compares the extracted `FINAL ANSWER` with the case's frozen numeric value. It can fail
even when the router and tool/code evaluators pass.


In [ ]:
answer_eval = check_numeric(example_case, example_result)
answer_eval["label"] = "correct" if answer_eval["numeric_ok"] else "incorrect"
answer_eval["score"] = 1 if answer_eval["numeric_ok"] else 0
answer_score = float(answer_eval["score"])
print(answer_eval)
print(f"Answer score: {answer_score:.0f}/1")


#### Combine component scores for this case

The teaching score below is the average of the three applicable evaluators. The release-style
`PASS` remains stricter: every required check must pass. Suite/experiment overall score is the
fraction of cases that pass all their required checks.


In [ ]:
component_scores = [router_score, tool_score, answer_score]
case_overall_score = sum(component_scores) / len(component_scores)
case_pass = all(score == 1.0 for score in component_scores)
print(f"Component-average score: {case_overall_score:.0%}")
print(f"Strict case result: {'PASS' if case_pass else 'FAIL'}")


---
## 5. Run a prompt experiment

An experiment changes one thing and holds the rest fixed: **same questions, model,
temperature, and evaluators**. The variant below intentionally relaxes tool-use and
precision guidance. Run it, compare the scores, then inspect the corresponding traces.

Each variant is recorded in Phoenix as a labeled parent span with its case IDs, evaluator
averages, and overall pass rate. Its agent traces are children of that span, so you can compare
the score summary and inspect the underlying calls in one place.


In [ ]:
CANDIDATE_SYSTEM_PROMPT = CREATIVE_SYSTEM_PROMPT
CANDIDATE_LABEL = f"{RUN_TAG}/prompt-candidate"
print("Candidate system prompt:\n", CANDIDATE_SYSTEM_PROMPT)

candidate = run_phoenix_experiment(
    CANDIDATE_LABEL,
    CANDIDATE_SYSTEM_PROMPT,
    STARTER_CASES,
    temperature=0.0,  # held fixed to isolate the prompt change
)
display(candidate[[c for c in ("case", "router_ok", "tools_used", "numeric_ok", "judge_pass", "PASS") if c in candidate]])


In [ ]:
comparison = baseline[["case", "PASS", "router_ok"]].merge(
    candidate[["case", "PASS", "router_ok"]], on="case", suffixes=("_baseline", "_candidate")
)
comparison["regressed"] = comparison["PASS_baseline"] & ~comparison["PASS_candidate"]
print(f"baseline overall: {baseline['PASS'].mean():.0%} ({int(baseline['PASS'].sum())}/{len(baseline)})")
print(f"candidate overall: {candidate['PASS'].mean():.0%} ({int(candidate['PASS'].sum())}/{len(candidate)})")
comparison


### Read the evaluator summary

Each component score is the fraction of applicable cases that passed that evaluator. The
**overall score** is the fraction of cases passing every required check. This mirrors L11's
per-evaluator experiment summary while keeping the strict all-checks pass rate visible.
The same component averages and overall score are attached to each parent experiment span in
Phoenix, in addition to the per-case annotations.


In [ ]:
experiment_summary = pd.DataFrame([
    {"variant": run["label"], **run["scores"]}
    for run in EXPERIMENT_RUNS
])
experiment_summary


### Optional: compare two router models instead

Keep the hardened prompt and the same case fixed, then change only the router model. The
workshop gateway is configured for `gpt-4o-mini` and `gpt-4o`; the larger model may use more
of your seat's budget. This agent also makes a separate code-generation call inside its data
tool; that inner call stays on `gpt-4o-mini`, so this is a router-model comparison, not a
whole-agent model comparison. It uses one numeric case to keep the optional exercise short.
Check the gateway's currently enabled model aliases before changing these names.


In [ ]:
MODEL_CASES = [next(c for c in GOLDEN["cases"] if c["id"] == "total_expenses_october")]
MODEL_A = AGENT_MODEL
MODEL_B = "gpt-4o"
model_a_label = f"{RUN_TAG}/model-{MODEL_A}"
model_b_label = f"{RUN_TAG}/model-{MODEL_B}"

model_a = run_phoenix_experiment(model_a_label, HARDENED_SYSTEM_PROMPT, MODEL_CASES, model=MODEL_A)
model_b = run_phoenix_experiment(model_b_label, HARDENED_SYSTEM_PROMPT, MODEL_CASES, model=MODEL_B)
display(model_a[["case", "PASS", "latency_s"]].merge(
    model_b[["case", "PASS", "latency_s"]], on="case", suffixes=(f"_{MODEL_A}", f"_{MODEL_B}")
))


---
## 6. Inspect and label a failure in Phoenix

Open the shared board, select `01-Part-3-evaluations`, and filter to your `session.id` and
the run-label prefix shown below. Open a failed candidate trace (or a baseline failure if the
candidate happens to pass). Compare the user question, router decision, tool input/output,
generated code, and final answer. Which component failed?

If the board's annotation UI is enabled for participants, add a **human** failure label and a
short evidence-based note. Otherwise, record the same note in the next cell. Do not classify a
case as a model failure until you've checked that the evaluator and expected answer are sound.


In [ ]:
print("Board:", PHOENIX_BOARD_URL)
print("Project:", PHOENIX_EVAL_PROJECT)
print("Seat filter (session.id):", SEAT)
failed_candidate_ids = candidate.loc[~candidate["PASS"], "case"].tolist()
review_ids = failed_candidate_ids or baseline.loc[~baseline["PASS"], "case"].tolist()
REVIEW_LABEL = CANDIDATE_LABEL if failed_candidate_ids else BASELINE_LABEL
print("Trace-name prefix:", REVIEW_LABEL + "/")
print("Candidate failures:", failed_candidate_ids or "none this run")
print("Review these case(s):", review_ids or "no failed cases; inspect any trace and challenge the evaluator")


In [ ]:
# Complete this after inspecting the trace. Keep the evidence concise and reproducible.
failure_note = {
    "run_label": REVIEW_LABEL,
    "case_id": "replace-with-failed-case-id",
    "failure_type": "router | skill | data | evaluator | prompt",
    "trace_evidence": "Which span/input/output shows the failure?",
    "expected_behavior": "What should the agent have done?",
    "reproducible": "yes | no | not_checked",
}
failure_note


---
## 7. Promote a reviewed failure to the golden dataset

Only promote a case when the failure is reproducible, the expected behavior is clear, and
ground truth can be independently checked. Capture at least: a unique `id`, the exact user
`query`, case `type`, expected behavior/value, tolerance (for numeric cases), and the rule for
computing ground truth.

For this repository, golden cases are generated by `scripts/generate_dataset.py`; don't edit
the JSON by hand. Add the case to `build_cases()` and compute its expected value from the
synthetic data, then regenerate and verify with `python scripts/generate_dataset.py --check`.
The notebook note above is a review candidate, not an automatic dataset change.


## Attach evaluator scores to the experiment traces

Each agent trace gets separate `Router Eval`, `Tool Eval`, `Ground Truth Eval` (or `LLM Judge
Eval`), and `Overall Case Eval` annotations. Each experiment's parent span gets aggregate
router, tool, answer, and **overall pass-rate** scores. In Phoenix, open the parent span named
`experiment/<run label>` to see the overall score and component-score attributes; use the
named evaluations to compare individual traces and variants. Human review remains separate.


In [ ]:
import pandas as pd

spans = phoenix_client.spans.get_spans(project_identifier=PHOENIX_EVAL_PROJECT, limit=2000)
span_ids = {}
for span in spans:
    attributes = span.get("attributes") or {}
    if attributes.get("session.id") != SEAT:
        continue
    span_name = span.get("name")
    span_id = (span.get("context") or {}).get("span_id")
    if span_name and span_id:
        span_ids[span_name] = span_id

annotation_groups = {}


def add_annotation(name, span_id, score, label, explanation):
    if span_id is None or score is None or pd.isna(score):
        return
    annotation_groups.setdefault(name, []).append({
        "span_id": span_id,
        "label": label,
        "score": float(score),
        "explanation": explanation,
    })


experiment_annotation_names = {
    "router": "Experiment Router Score",
    "tool": "Experiment Tool Score",
    "answer": "Experiment Answer Score",
    "overall": "Overall Experiment Score",
}
case_annotation_names = {
    "router": "Router Eval",
    "tool": "Tool Eval",
    "numeric": "Ground Truth Eval",
    "judge": "LLM Judge Eval",
    "overall": "Overall Case Eval",
}

for run in EXPERIMENT_RUNS:
    label = run["label"]
    scorecard = run["scorecard"]
    for component, annotation_name in experiment_annotation_names.items():
        score = run["scores"].get(component)
        if score is not None:
            add_annotation(
                annotation_name,
                run["span_id"],
                score,
                f"{score:.0%}",
                f"{run['passed']}/{run['total']} cases pass all required checks; {component} evaluator average.",
            )

    for row in scorecard.to_dict(orient="records"):
        span_id = span_ids.get(f"{label}/{row['case']}")
        add_annotation(
            case_annotation_names["router"], span_id,
            float(bool(row["router_ok"])),
            "correct" if row["router_ok"] else "incorrect",
            f"Required tools: {REQUIRED_TOOLS.get(row['case'], [])}; used: {row.get('tools_used', '')}",
        )

        n_chunks = row.get("n_chunks")
        if pd.notna(n_chunks) and n_chunks > 0:
            tool_ok = bool(row.get("code_valid")) and bool(row.get("no_traceback"))
            add_annotation(
                case_annotation_names["tool"], span_id, float(tool_ok),
                "runnable" if tool_ok else "not_runnable",
                f"Generated chunks={int(n_chunks)}; valid={row.get('code_valid')}; no_traceback={row.get('no_traceback')}",
            )

        numeric_ok = row.get("numeric_ok")
        if pd.notna(numeric_ok):
            add_annotation(
                case_annotation_names["numeric"], span_id, float(numeric_ok),
                "correct" if numeric_ok else "incorrect",
                f"Extracted={row.get('extracted')}; tolerance-checked against golden ground truth.",
            )
        judge_pass = row.get("judge_pass")
        if pd.notna(judge_pass):
            add_annotation(
                case_annotation_names["judge"], span_id, float(judge_pass),
                "pass" if judge_pass else "fail",
                f"{label}: behavior/advisory rubric result.",
            )

        passed = bool(row["PASS"])
        add_annotation(
            case_annotation_names["overall"], span_id, float(passed),
            "PASS" if passed else "FAIL",
            f"{label}: all required checks {'passed' if passed else 'did not pass'}.",
        )

if annotation_groups:
    for annotation_name, rows in annotation_groups.items():
        phoenix_client.spans.log_span_annotations_dataframe(
            dataframe=pd.DataFrame(rows),
            annotation_name=annotation_name,
            annotator_kind="CODE",
        )
        values = [row["score"] for row in rows]
        print(f"{annotation_name}: {sum(values) / len(values):.0%} average across {len(values)} spans")
    print("Named evaluator scores and overall experiment scores are logged to Phoenix")
else:
    print("No matching spans found yet; rerun this cell after the traces arrive")


---
## Takeaways

- Router, execution, answer-quality, and ground-truth checks catch different failures.
- A prompt or model change is an experiment only when you compare it on the same cases with
  the same evaluators and hold other variables fixed.
- Phoenix traces help explain *why* a score changed; human review decides whether a failure
  is real and worth adding to the golden set.
- The complete golden suite and regression gate remain in `scripts/run_evals.py` and CI.

For a full-cost run of all 11 cases, use the harness outside the short training path or during
an instructor-led extension.


---
### Workshop home

[Return to the workshop README](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/README.md)
